<a href="https://colab.research.google.com/github/AkhileshSR/AkhileshSR/blob/main/102025_11_RAG_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation (RAG) from Scratch

In this exercise, we'll build a **simple RAG pipeline** using:
- OpenAI embeddings  
- FAISS vector database  
- OpenAI LLM for answer generation  

No LangChain — just the core logic.


In [ ]:
!pip install openai faiss-cpu --quiet

In [ ]:
import os
from getpass import getpass
import numpy as np
import faiss
from openai import OpenAI

# 🔑 Enter your OpenAI API Key
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI()

Enter your OpenAI API key: ··········



## 1️⃣ Create a Small Knowledge Base
Define a few sentences that represent what our AI "knows".


In [ ]:
docs = [
    "Virat Kohli is an Indian cricketer and former captain of the national team.",
    "The capital of France is Paris.",
    "Python is a popular programming language for data science.",
    "Pizza originated in Italy and is a popular Italian dish.",
    "The Moon is Earth's only natural satellite.",
    "Favorite dish of Sridhar is noodles"
]


## 2️⃣ Create Embeddings for Each Document
Convert each sentence into an embedding vector that captures its meaning.


In [ ]:
embeddings = []
for doc in docs:
    response = client.embeddings.create(
        input=doc,
        model="text-embedding-3-small"
    )
    embeddings.append(response.data[0].embedding)
    print(doc)
    print(response.data[0].embedding)

Virat Kohli is an Indian cricketer and former captain of the national team.
[0.0004379896854516119, 0.061106614768505096, 0.001033069915138185, -0.006147298496216536, -0.01728741265833378, -0.0028649051673710346, 0.04106719046831131, 0.01802866719663143, -0.0015932706883177161, 0.025015201419591904, 0.010036753490567207, 0.04018109291791916, -0.006117477547377348, 0.023515652865171432, 0.06291288882493973, -0.007659627590328455, -0.03029770404100418, 0.011033612303435802, -0.003248312510550022, 0.014126432128250599, 0.05033712834119797, -0.006189899053424597, 0.005150439217686653, -0.015123290941119194, -0.01772194169461727, 0.0255434513092041, -0.030434025451540947, 0.009943031705915928, 0.038136254996061325, -0.008997293189167976, 0.011689664795994759, -0.00832420028746128, -0.023106684908270836, -0.027724614366889, 0.023515652865171432, 0.0001328613143414259, -0.011621504090726376, -0.025594573467969894, -0.002131106099113822, 0.02968425117433071, -0.0011874978663399816, -0.03581877

In [ ]:
embedding_matrix = np.array(embeddings).astype("float32")
print("✅ Created embeddings for", len(docs), "documents.")
print(embedding_matrix)

✅ Created embeddings for 6 documents.
[[ 0.00043799  0.06110661  0.00103307 ...  0.03017842  0.00359764
   0.00984079]
 [ 0.03112892  0.02981416  0.02432309 ... -0.01688889  0.01733359
   0.0194024 ]
 [ 0.01480796 -0.02266525  0.021362   ... -0.04514162  0.0166684
   0.02179642]
 [-0.05448304 -0.01913278 -0.03467454 ... -0.00420158  0.04432781
   0.00408575]
 [ 0.00772781  0.03247659  0.00208872 ... -0.01636088  0.00410908
  -0.00749677]
 [-0.01283801 -0.05274539 -0.0226866  ...  0.02240793  0.02040654
  -0.02733539]]



## 3️⃣ Store Embeddings in FAISS (Vector Database)
FAISS lets us search by *meaning similarity* instead of exact match.


In [ ]:
index = faiss.IndexFlatL2(embedding_matrix.shape[1])
print("✅ Created an index in FAISS vector store.")

✅ Created an index in FAISS vector store.


In [ ]:
index.add(embedding_matrix)
print("✅ Added all documents to FAISS vector store.")

✅ Added all documents to FAISS vector store.



## 4️⃣ Write a Retrieval Function
Retrieve the top-k most relevant sentences for a given query.


In [ ]:
#In this query is the query, k is the top k chunks to return
def retrieve(query, k=2):
    # step 1 - convert the query into embedding
    q_emb = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    ).data[0].embedding

    # step 2 - create a embedding matrix using numpy
    query_vector = np.array([q_emb]).astype("float32")

    # step 3 - search inside FAISS for the given embeddings of the query
    distances, indices = index.search(query_vector, k)

    # step 4 - return the top k chunks with distance value
    return [(docs[i], distances[0][j]) for j, i in enumerate(indices[0])]

In [ ]:
query = "Who is Virat Kohli?"
results = retrieve(query, 5)

In [ ]:
# print the results with chunks and distance
for r, d in results:
    print(f"- {r} (distance={d:.3f})")
# Smallest distance means similar

- Virat Kohli is an Indian cricketer and former captain of the national team. (distance=0.541)
- The capital of France is Paris. (distance=1.704)
- Favorite dish of Sridhar is noodles (distance=1.760)
- Pizza originated in Italy and is a popular Italian dish. (distance=1.898)
- Python is a popular programming language for data science. (distance=1.971)


In [ ]:
query = "Who is Virat Kohli?"
results = retrieve(query, 2)

In [ ]:
# print the results with chunks and distance
for r, d in results:
    print(f"- {r} (distance={d:.3f})")
# Smallest distance means more similar

- Virat Kohli is an Indian cricketer and former captain of the national team. (distance=0.541)
- The capital of France is Paris. (distance=1.704)


In [ ]:
query = "Who is Virat Kohli?"
results = retrieve(query, 3)

In [ ]:
# print the results with chunks and distance
for r, d in results:
    print(f"- {r} (distance={d:.3f})")
# Smallest distance means similar


## 5️⃣ Write a Simple RAG Function
Now we combine **retrieval + generation** to produce factual answers.


In [ ]:
def generate_answer(query, k=2):
    # retrieve top k chunks
    results = retrieve(query, k=k)

    # combine the k results
    context_text = "\n".join([r[0] for r in results])

    # prompt combining the query and the results retrieved from vector store
    prompt = f"""
    Use the following context to answer the question accurately.

    Context:
    {context_text}

    Question: {query}
    Answer:
    """

    #comment this line if you dont want to see the prompt
    print(f"""The final prompt is prompt: {prompt}""")

    # make an LLM API call to get the answer
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return completion.choices[0].message.content.strip()



## 6️⃣ Try It Out!
Test the RAG pipeline with a few questions.


In [ ]:
query = "Who is Virat Kohli?"
print(generate_answer(query, k=3))

The final prompt is prompt: 
    Use the following context to answer the question accurately.

    Context:
    Virat Kohli is an Indian cricketer and former captain of the national team.
The capital of France is Paris.
Favorite dish of Sridhar is noodles

    Question: Who is Virat Kohli?
    Answer:
    
Virat Kohli is an Indian cricketer and former captain of the national team.


In [ ]:
query = "What is the capital of france?"
print(generate_answer(query, 2))

The final prompt is prompt: 
    Use the following context to answer the question accurately.

    Context:
    The capital of France is Paris.
Virat Kohli is an Indian cricketer and former captain of the national team.

    Question: What is the capital of france?
    Answer:
    
The capital of France is Paris.


In [ ]:
query = "What is the favorite dish of Sridhar?"
print(generate_answer(query, 3))

The final prompt is prompt: 
    Use the following context to answer the question accurately.

    Context:
    Favorite dish of Sridhar is noodles
Pizza originated in Italy and is a popular Italian dish.
Virat Kohli is an Indian cricketer and former captain of the national team.

    Question: What is the favorite dish of Sridhar?
    Answer:
    
The favorite dish of Sridhar is noodles.


In [ ]:
queries = [
    "Who is Virat Kohli?",
    "Where did pizza come from?",
    "What is the capital of France?",
    "What programming language is good for data science?",
    "What is Earth's natural satellite?",
    "What is favorite dish of Sridhar"
]

for q in queries:
    print(f"Q: {q}")
    print("A:", generate_answer(q))
    print("-" * 60)


Q: Who is Virat Kohli?
The final prompt is prompt: 
    Use the following context to answer the question accurately.

    Context:
    Virat Kohli is an Indian cricketer and former captain of the national team.
The capital of France is Paris.

    Question: Who is Virat Kohli?
    Answer:
    
A: Virat Kohli is an Indian cricketer and former captain of the national team.
------------------------------------------------------------
Q: Where did pizza come from?
The final prompt is prompt: 
    Use the following context to answer the question accurately.

    Context:
    Pizza originated in Italy and is a popular Italian dish.
Favorite dish of Sridhar is noodles

    Question: Where did pizza come from?
    Answer:
    
A: Pizza originated in Italy.
------------------------------------------------------------
Q: What is the capital of France?
The final prompt is prompt: 
    Use the following context to answer the question accurately.

    Context:
    The capital of France is Paris.
Vi

## Exercise 1 — Retrieval Only (See the “lookup”)

**Do**

* Run:

  ```python
  retrieve("Where did pizza come from?", k=3)
  ```
* Print chunks + distances.

**Observe**

* Correct sentence retrieved: *“Pizza originated in Italy…”*
* System is **searching by meaning**, not answering.
* This is the **memory access step**.

---

## Exercise 2 — Retrieval + Generation (RAG)

**Do**

* Run:

  ```python
  generate_answer("Where did pizza come from?", k=2)
  ```

**Observe**

* Answer is accurate and grounded.
* Answer content comes **only from retrieved text**.
* This is **retrieve → then generate**.

---

## Exercise 3 — Turn Retrieval Off

**Do**

* Bypass retrieval and ask LLM directly:

  > “Where did pizza come from?”

**Observe**

* Generic or guessed answer.
* No grounding.
* Clear contrast:
  **Without retrieval → guessing**
  **With retrieval → knowing**


## 🧪 Exercise 4
- Add a new document to your knowledge base and rebuild FAISS.  
- Rephrase queries and see if retrieval still works.  
- Adjust `k` to see how many context pieces affect the answer.



## 🧠 Reflection
- Which part of this acts as *memory*?  
- How does retrieval improve factual accuracy?  
- How could you extend this to your company’s private knowledge base?
